# 02 Provider Routing and Failover (OpenClaw, 2026)

## What This Lesson Is
Build policy-based model routing and failure recovery using OpenClaw primary/fallback patterns.

## Scientific Lens
- Concept: Model failover policy must be explicit, testable, and cost-aware.
- Measure: Route correctness and fallback success under injected provider failures.
- Validity Limit: Failover logic cannot compensate for globally degraded models or incorrect task classification.


## How It Works
1. Define routing policy by task criticality and cost envelope.
2. Simulate provider auth/availability failures and verify fallback order.
3. Apply policy in OpenClaw config and validate with `models status` and `models set`.


In [ ]:
import os
import shutil

HAS_OPENCLAW = shutil.which("openclaw") is not None
print("openclaw available:", HAS_OPENCLAW)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("OLLAMA_BASE_URL:", os.getenv("OLLAMA_BASE_URL", "<unset>"))


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: uses real OpenClaw integration commands and skips gracefully if prerequisites are missing.


In [ ]:
# Deterministic Demo
tasks = [
    {"id":"T1","type":"critical_code","max_cost":0.12},
    {"id":"T2","type":"summary","max_cost":0.02},
    {"id":"T3","type":"private","max_cost":0.03},
]
policy = {
    "critical_code": ["openai/gpt-5.1-codex", "anthropic/claude-sonnet-4-5"],
    "summary": ["ollama/qwen2.5-coder:1.5b", "openai/gpt-4.1-mini"],
    "private": ["venice/llama-3.3-70b", "ollama/qwen2.5-coder:1.5b"],
}
provider_up = {"openai": False, "anthropic": True, "ollama": True, "venice": True}

def select(task):
    for model in policy[task["type"]]:
        provider = model.split("/",1)[0]
        if provider_up.get(provider, False):
            return model
    return None

routes = {t["id"]: select(t) for t in tasks}
print(routes)
assert routes["T1"] == "anthropic/claude-sonnet-4-5"
assert routes["T2"].startswith("ollama/")


In [ ]:
# Live Demo
import os, shutil, subprocess

if not HAS_OPENCLAW:
    print("Skipping live routing demo: openclaw CLI not installed.")
else:
    primary = os.getenv("OPENCLAW_PRIMARY_MODEL", "openai/gpt-5.1-codex")
    fallback = os.getenv("OPENCLAW_FALLBACK_MODEL", "ollama/qwen2.5-coder:1.5b")
    for cmd in [
        ["openclaw", "models", "set", primary],
        ["openclaw", "models", "fallbacks", "add", fallback],
        ["openclaw", "models", "status"],
    ]:
        print("$", " ".join(cmd))
        p = subprocess.run(cmd, capture_output=True, text=True)
        print((p.stdout or p.stderr).strip()[:1400])


## Applied Labs
1. Add cooldown-aware provider state and prove your selector avoids flapping providers.
2. Introduce auth-profile rotation simulation within one provider before cross-provider fallback.
3. Create a cost-budget guard that blocks expensive fallback models for low-priority tasks.

## Validation Checklist
- Fallback order is deterministic and asserted.
- Routing policy is workload-driven, not ad-hoc prompt-based.
- Live config changes are observable via `models status`.

## Further Reading
- OpenClaw model failover concepts: https://docs.openclaw.ai/models
- Model provider concepts: https://docs.openclaw.ai/concepts/model-providers
- Provider quickstart: https://docs.openclaw.ai/providers/models
